# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen4107/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

## My data contract

**Unit of analysis:** One row represents the daily performance of one content page for one client on one reporting date.

**Tables used:** `fact_content_daily_performance`

**Time window:** March 2026 (`month=2026-03`)

**Prediction / ranking:** I will rank content pages by refresh opportunity using a declining-performance proxy.

**Deliberately excluded:** Future performance metrics and any label-derived columns are excluded to prevent data leakage.

In [19]:
!pip -q install duckdb huggingface_hub

import duckdb
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

grain = con.sql("""
SELECT
  report_date,
  client_hash_id,
  content_hash_id,
  gsc_impressions AS impressions,
  gsc_clicks AS clicks
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 5
""").df()

grain

,report_date,client_hash_id,content_hash_id,impressions,clicks
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0


## 2. Fields: feature / label / context / excluded

| Bucket | Fields |
|---|---|
| **Features** | `gsc_impressions`, `gsc_clicks`, `gsc_ctr`, `gsc_avg_position`, `ga4_sessions` |
| **Label / Proxy** | Declining performance (used later as a refresh-opportunity proxy) |
| **Context** | `report_date`, `client_hash_id`, `content_hash_id` |
| **Excluded** | Future metrics and any label-derived fields because they leak the correct answer into the model. |

### Why these five features?

- **Impressions:** Knowable after the reporting day because Search Console records visibility.
- **Clicks:** Available from the same reporting day.
- **CTR:** Calculated from impressions and clicks already observed.
- **Average position:** Reported by Search Console on that day.
- **Sessions:** Available only when GA4 data has synced.

In [20]:
features = con.sql("""
SELECT
  content_hash_id,
  report_date,
  gsc_impressions,
  gsc_clicks,
  CASE
    WHEN gsc_impressions > 0
    THEN ROUND((gsc_clicks * 100.0) / gsc_impressions, 2)
    ELSE 0
  END AS ctr,
  gsc_avg_position,
  ga4_sessions
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
LIMIT 10
""").df()

features

,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,ga4_sessions
0,content_09be8cc7fcb222af,2026-03-01,0,0,0.0,NaN,1
1,content_851afac9fe13612e,2026-03-01,0,0,0.0,NaN,1
2,content_cee6c6fc8c51af14,2026-03-01,0,0,0.0,NaN,1
3,content_5e120e972f11f833,2026-03-01,0,0,0.0,NaN,1
4,content_16a7291bb6ecaebe,2026-03-01,0,0,0.0,NaN,1
5,content_5c80451459c29b4a,2026-03-01,5,0,0.0,5.400000,1
6,content_9f9a23189cb8f12f,2026-03-01,0,0,0.0,NaN,1
7,content_4ab81290aec524dd,2026-03-01,0,0,0.0,NaN,1
8,content_b1f61fc81b28b2d4,2026-03-01,39,0,0.0,5.666667,2
9,content_e25ea7297a1dffd3,2026-03-01,179,0,0.0,5.156425,2


## 3. Verify it with queries (grain, counts, missing values, windows)

The following three queries verify my data contract:

1. Confirm the row count and date span.
2. Confirm the number of unique content pages in the slice.
3. Confirm data availability using `IS TRUE`.

In [21]:
# Query 1 — row count + date span
q1 = con.sql("""
SELECT
  COUNT(*) AS row_count,
  MIN(report_date) AS start_date,
  MAX(report_date) AS end_date
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print("Query 1")
print(q1)

# Query 2 — unique pages
q2 = con.sql("""
SELECT
  COUNT(DISTINCT content_hash_id) AS unique_pages
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print("\nQuery 2")
print(q2)

# Query 3 — availability
q3 = con.sql("""
SELECT
  COUNT(*) AS available_rows
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
""").df()

print("\nQuery 3")
print(q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1
   row_count start_date   end_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 2
   unique_pages
0        331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Query 3
   available_rows
0          413966


## 4. Data limits

### Limitation

This notebook uses only the March 2026 warehouse partition, so it does not capture long-term seasonality or future recovery. The results should be interpreted as decision-support rather than proof that refreshing content will increase traffic.

### Leakage experiment

To demonstrate data leakage, I intentionally created a label from CTR and then included CTR as a feature. This produced an unrealistically high score because the feature directly contained the answer. After observing the effect, the leaked feature was removed.

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Build the dataset and CREATE ctr
data = con.sql("""
SELECT
    gsc_impressions,
    gsc_clicks,
    CASE
        WHEN gsc_impressions > 0
        THEN (gsc_clicks * 100.0) / gsc_impressions
        ELSE 0
    END AS ctr,
    gsc_avg_position,
    ga4_sessions
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE ga4_data_available IS TRUE
AND gsc_impressions > 0
LIMIT 5000
""").df()

# Create proxy label
data["low_ctr_label"] = (data["ctr"] < 0.5).astype(int)

# Honest model (WITHOUT ctr)
X = data[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_sessions"]]
y = data["low_ctr_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)
honest_acc = accuracy_score(y_test, model.predict(X_test))

# Leaked model (WITH ctr)
X_leak = data[["gsc_impressions", "gsc_clicks", "gsc_avg_position", "ga4_sessions", "ctr"]]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak, y, test_size=0.2, random_state=42
)

leak_model = LogisticRegression(max_iter=500)
leak_model.fit(X_train, y_train)
leaked_acc = accuracy_score(y_test, leak_model.predict(X_test))

print("Honest accuracy:", round(honest_acc, 3))
print("Leaked accuracy:", round(leaked_acc, 3))

Honest accuracy: 0.99
Leaked accuracy: 0.998


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.